In [4]:
import os
import pickle
import json
import numpy as np
from matplotlib import pyplot as plt

In [5]:
def get_fluent_key(fluent_value: int):
    for k in log_goals.keys():
        if log_goals[k] == fluent_value:
            return k
    return None


def get_mutex_fluent(fluent_value: int):
    fluent = get_fluent_key(fluent_value)
    fluent = fluent.rsplit(' ', 1)
    mutex_values = set()
    for k in log_goals.keys():
        if k.startswith(fluent[0]):
            mutex_values.add(log_goals[k])
        # if k.endswith(fluent[1]):
        #     mutex_values.add(goals[k])
    return mutex_values

def compute_score(predictions: list, goal: list):
    score = 0
    for g in goal:
        score += predictions[g]
    return score

def compute_scores(predictions: list, goal: list):
    scores = []
    for g in goal:
        scores.append(compute_score(predictions, g))
    return scores

In [6]:
def get_fluent_key(fluent_value: int):
    for k in log_goals.keys():
        if log_goals[k] == fluent_value:
            return k
    return None

def get_mutex_fluent_depots(fluent_value: int):
    fluent = get_fluent_key(fluent_value)
    fluent = fluent.rsplit(' ', 1)
    mutex_values = set()
    for k in log_goals.keys():
        if k.startswith(fluent[0]):
            mutex_values.add(log_goals[k])
        if k.endswith(fluent[1]):
            mutex_values.add(log_goals[k])
    return mutex_values

def get_mutex_fluent(fluent_value: int):
    fluent = get_fluent_key(fluent_value)
    fluent = fluent.rsplit(' ', 1)
    mutex_values = set()
    for k in log_goals.keys():
        if k.startswith(fluent[0]):
            mutex_values.add(log_goals[k])
        # if k.endswith(fluent[1]):
        #     mutex_values.add(goals[k])
    return mutex_values

def get_mutex_fluent_satellite(fluent_value: int):
    fluent = get_fluent_key(fluent_value)
    fluent = fluent.rsplit(' ', 1)
    mutex_values = set()
    if fluent[0].startswith("POINTING"):
        for k in log_goals.keys():
            if k.startswith(fluent[0]):
                mutex_values.add(log_goals[k])
    return mutex_values

def get_mutex_fluent_blocksworld(fluent_value: int):
    fluent = get_fluent_key(fluent_value)
    fluent = fluent.split(' ')
    mutex_values = set()
    for k in log_goals.keys():
        if fluent[0] == "ON":
            if k.startswith("ON "):                    
                obj=k.split(' ')
                if obj[1] == fluent[1]:
                    mutex_values.add(log_goals[k])
                if obj[2] == fluent[2]:
                    mutex_values.add(log_goals[k])
                obj=k.split(' ')
            if k.startswith("ON-TABLE ") or k.startswith("ONTABLE "):
                if "ON " in k and k.endswith(fluent[2]):
                    mutex_values.add(log_goals[k])
                    
            if k.startswith("CLEAR "):
                if k.startswith(f"ON {fluent[1]}"):
                    mutex_values.add(log_goals[k])
                    
                    
        if fluent[0] == "ON-TABLE" or fluent[0] == "ONTABLE":
            if k.startswith("ON "):
                obj=k.split(' ')
                if obj[1] == fluent[1]:
                    mutex_values.add(log_goals[k])
        if fluent[0] == "CLEAR":
            if k.startswith("ON "):
                obj=k.split(' ')
                if obj[2] == fluent[1]:
                    mutex_values.add(log_goals[k])

    return mutex_values

def get_mutex_goal(fluent_values: list):
    mutex_values = set()
    for f in fluent_values:
        mutex_values = mutex_values.union(get_mutex_fluent_blocksworld(f))
    return mutex_values

def compute_score(predictions: list, goal: list):
    score = 0
    for g in goal:
        score += predictions[g]
    return score

def compute_scores(predictions: list, goal: list):
    scores = []
    for g in goal:
        scores.append(compute_score(predictions, g))
    return scores

In [7]:
def get_indexes(prediction: list, threshold: float = 0.4):
    indexes = []
    for i, p in enumerate(prediction):
        if p > threshold:
            indexes.append(i)
    return indexes

In [8]:
from tqdm import tqdm
domains=["depots", "satellite", "blocksworld", "logistics", "driverlog", "zenotravel"]

for domain in domains:
    nome_pereira=domain.replace('blocksworld', 'block-words').replace('zenotravel', 'zeno-travel')
    diz_goal_path=f"../../GRNet/code/dictionaries/{domain}/dizionario_goal" #Occhio che il dizionario goal è diverso da quello precedente
    res_path=f"../../goal_recognition/results/explain/{domain}/ris_final_11.json" #Path del json generato da compute_results.ipynb
    res_gr= f"../../goal_recognition/results/explain/{domain}/ris_final_gr.json" 
    with open(diz_goal_path, 'rb') as f:
        log_goals = pickle.load(f)
    with open(res_path, 'r') as f:
        results = json.load(f)
    with open(res_gr, 'r') as f:
        results_gr = json.load(f)
    perc_tot = np.zeros((10,))
    tau1=0.4
    tau2=0.1
    tot_scores={}
    for k in tqdm(results_gr.keys()):
        if domain in k or nome_pereira in k:

            
            count = 0
            correct = 0
            cv = 0
            
            prob_dict_gr = results_gr[k]
            goals_list_gr = results_gr[k]['possible_goals']
            pred_dict_gr = results_gr[k]['predictions']
            
            prob_dict = results[k]
            goals_list = results[k]['possible_goals']
            pred_dict = results[k]['predictions']

            tot_scores[k] = {}
            tot_scores[k]['Possible Goals'] = goals_list
            tot_scores[k]['True Goal'] = prob_dict['correct_goal']
            tot_scores[k]['TD'] = []
            tot_scores[k]['GRNET'] = []
            
            if prob_dict['correct_goal'] != prob_dict_gr['correct_goal']:
                print('Error')
                break
            
            sum = np.zeros((len(pred_dict['0']),))
            next_perc = 0
            
            sum_gr = np.zeros((len(pred_dict_gr['0']),))
            for i in prob_dict_gr['init_state']:
                sum_gr[i] = tau2
        
            for i in prob_dict['init_state']:
                sum[i] = tau2
                
            scores_matrix = np.zeros((len(pred_dict.keys())+1, len(goals_list)))
            scores_matrix_gr = np.zeros((len(pred_dict_gr.keys())+1, len(goals_list_gr)))
            scores_matrix[0] = compute_scores(sum, goals_list)
            scores_matrix_gr[0] = compute_scores(sum_gr, goals_list_gr)
            
            for j, p in enumerate(pred_dict.keys()):
                #create predictions as the max of the two predictions for each value
                predictions_gr = pred_dict_gr[p]
                predictions = pred_dict[p]
                
                predictions = [log if  log > 0.05 else 0 for log in predictions]
                predictions_gr = [log if  log > 0.05 else 0 for log in predictions_gr]
                sum += predictions
                indexes = get_indexes(predictions, threshold=tau1)
                if len(indexes) > 0:
                    for i in indexes:
                        mutex = [m for m in get_mutex_fluent(i) if m not in indexes]
                        for m in mutex:
                            sum[m] = 0
                if len(indexes) > 0:
                    for i in indexes:
                        if domain=="depots":
                            mutex = [m for m in get_mutex_fluent_depots(i) if m not in indexes]
                        elif domain=="satellite":
                            mutex = [m for m in get_mutex_fluent_satellite(i) if m not in indexes]
                        elif domain=="blocksworld":
                            mutex = [m for m in get_mutex_fluent_blocksworld(i) if m not in indexes]
                        else:
                            mutex = [m for m in get_mutex_fluent(i) if m not in indexes]
                        for m in mutex:
                            sum[m] = 0
                # scores = compute_scores(sum, goals_list)
                # scores_gr= compute_scores(predictions_gr, goals_list_gr)
                # scores_matrix[int(p)+1] = scores
                # scores_matrix_gr[int(p)+1] = scores_gr
            tot_scores[k]['TD'] = scores_matrix.tolist()
            tot_scores[k]['GRNET'] = scores_matrix_gr.tolist()
            
    with open('../../goal_recognition/results/explain/'+domain+'.json', 'w') as f:
        json.dump(tot_scores, f)

 90%|████████▉ | 847/944 [00:03<00:00, 226.72it/s]


KeyboardInterrupt: 

# Preparazione risultati

In [19]:
from tqdm import tqdm
domains=["blocksworld", "satellite", "depots", "logistics", "driverlog", "zenotravel"]

for domain in domains:
    nome_pereira=domain.replace('blocksworld', 'block-words').replace('zenotravel', 'zeno-travel')
    diz_goal_path=f"../../GRNet/code/dictionaries/{domain}/dizionario_goal" #Occhio che il dizionario goal è diverso da quello precedente
    res_path=f"../../goal_recognition/results/explain/{domain}/ris_verbose_14.json" #Path del json generato da compute_results.ipynb
    res_gr= f"../../goal_recognition/results/explain/{domain}/ris_verbose_gr.json" 
    with open(diz_goal_path, 'rb') as f:
        log_goals = pickle.load(f)
    with open(res_path, 'r') as f:
        results = json.load(f)
    with open(res_gr, 'r') as f:
        results_gr = json.load(f)
    perc_tot = np.zeros((10,))
    tau1=0.4
    tau2=0.1
    tot_scores={}
    for k in tqdm(list(results_gr.keys())):
        if domain in k or nome_pereira in k:

            
            count = 0
            correct = 0
            cv = 0
            
            prob_dict_gr = results_gr[k]
            goals_list_gr = results_gr[k]['possible_goals']
            pred_dict_gr = results_gr[k]['predictions']
            
            prob_dict = results[k]
            goals_list = results[k]['possible_goals']
            pred_dict = results[k]['predictions']

            tot_scores[k] = {}

            
            if prob_dict['correct_goal'] != prob_dict_gr['correct_goal']:
                print('Error')
                break
            
            sum = np.zeros((len(pred_dict[list(pred_dict.keys())[0]]),))
            next_perc = 0
            
            sum_gr = np.zeros((len(pred_dict_gr[list(pred_dict_gr.keys())[0]]),))
            for i in prob_dict_gr['init_state']:
                sum_gr[i] = tau2
        
            for i in prob_dict['init_state']:
                sum[i] = tau2
                
            scores_matrix = np.zeros((len(pred_dict.keys())+1, len(goals_list)))
            scores_matrix_gr = np.zeros((len(pred_dict_gr.keys())+1, len(goals_list_gr)))
            scores_matrix[0] = compute_scores(sum, goals_list)
            scores_matrix_gr[0] = compute_scores(sum_gr, goals_list_gr)
            
            for j, p in enumerate(pred_dict.keys()):
                #create predictions as the max of the two predictions for each value
                predictions_gr = pred_dict_gr[p]
                predictions = pred_dict[p]
                
                predictions = [log if  log > 0.05 else 0 for log in predictions]
                predictions_gr = [log if  log > 0.05 else 0 for log in predictions_gr]
                
                sum += predictions
                indexes = get_indexes(predictions, threshold=tau1)
                if len(indexes) > 0:
                    for i in indexes:
                        mutex = [m for m in get_mutex_fluent(i) if m not in indexes]
                        for m in mutex:
                            sum[m] = 0
                if len(indexes) > 0:
                    for i in indexes:
                        if domain=="depots":
                            mutex = [m for m in get_mutex_fluent_depots(i) if m not in indexes]
                        elif domain=="satellite":
                            mutex = [m for m in get_mutex_fluent_satellite(i) if m not in indexes]
                        elif domain=="blocksworld":
                            mutex = [m for m in get_mutex_fluent_blocksworld(i) if m not in indexes]
                        else:
                            mutex = [m for m in get_mutex_fluent(i) if m not in indexes]
                        for m in mutex:
                            sum[m] = 0
                # scores = compute_scores(sum, goals_list)
                # scores_gr= compute_scores(predictions_gr, goals_list_gr)
                # scores_matrix[int(p)+1] = scores
                # scores_matrix_gr[int(p)+1] = scores_gr
            # tot_scores[k]['TD'] = scores_matrix.tolist()
            # tot_scores[k]['GRNET'] = scores_matrix_gr.tolist()
                corr_goal=prob_dict['correct_goal']
                ris_parziali={}
                for fluent in corr_goal:
                    if fluent not in ris_parziali:
                        ris_parziali[fluent]={}
                        ris_parziali[fluent]['TD']=0
                        ris_parziali[fluent]['GR']=0
                    ris_parziali[fluent]['TD']=sum[fluent]
                    ris_parziali[fluent]['GR']=predictions_gr[fluent]
                tot_scores[k][p]={}
                for fluent in ris_parziali.keys():
                    for key, value in log_goals.items():
                        if value==fluent:
                            if key not in tot_scores[k]:
                                tot_scores[k][p][key]={}
                            tot_scores[k][p][key]=ris_parziali[fluent]
                        
                        
    #print(tot_scores)
              
    with open('../../goal_recognition/results/explain/'+domain+'/explain_14.json', 'w') as f:
        json.dump(tot_scores, f, indent=4)

100%|██████████| 944/944 [00:01<00:00, 489.88it/s]


# Calcolo rf su singolo fatto

In [20]:
import json
from tqdm import tqdm
domains=["blocksworld", "satellite", "depots", "logistics", "driverlog", "zenotravel"]

for domain in domains:
    print(domain)
    with open('../../goal_recognition/results/explain/'+domain+'/explain_14.json', 'r') as f:
        results = json.load(f)
    RF_TOT_TD=0
    RF_TOT_GR=0
    for problem in list(results.keys()):
        # print(problem)
        RF_PROB_TD=0
        RF_PROB_GR=0
        for action in results[problem].keys():
            # print(action)
            RF_ACTION_TD=0
            RF_ACTION_GR=0
            for fluent in results[problem][action].keys():
                # print(fluent)
                # print(results[problem][action][fluent]['TD'])
                if results[problem][action][fluent]['TD']>=0.4:
                    
                    RF_ACTION_TD+=1
                if results[problem][action][fluent]['GR']>=0.4:
                    RF_ACTION_GR+=1
            RF_PROB_TD+=RF_ACTION_TD/len(results[problem][action].keys())
            RF_PROB_GR+=RF_ACTION_GR/len(results[problem][action].keys())
        RF_TOT_TD+=RF_PROB_TD/len(results[problem].keys())
        RF_TOT_GR+=RF_PROB_GR/len(results[problem].keys())
    print(f"RF TD: {RF_TOT_TD/len(results.keys())}")
    print(f"RF GR: {RF_TOT_GR/len(results.keys())}")
        


blocksworld
RF TD: 0.22246676711410446
RF GR: 0.22780416050983382
satellite
RF TD: 0.33908305466540756
RF GR: 0.2608436737723752
depots
RF TD: 0.3069421057615147
RF GR: 0.242774791183698
logistics
RF TD: 0.2945062885692575
RF GR: 0.2125868840773971
driverlog
RF TD: 0.21022753210878817
RF GR: 0.3102607717324373
zenotravel
RF TD: 0.3362871522085229
RF GR: 0.28333957288220546


In [2]:
import pickle

with open("../../GRNet/code/dictionaries/logistics/dizionario_goal", 'rb') as f:
    diz_goal=pickle.load(f)
print(diz_goal)

{'AT OBJ77 POS12': 33, 'AT OBJ22 POS11': 26, 'AT OBJ13 POS44': 143, 'AT OBJ44 POS22': 129, 'AT OBJ33 POS11': 62, 'AT OBJ55 POS21': 69, 'AT OBJ12 POS33': 58, 'AT OBJ21 POS23': 7, 'AT OBJ77 POS33': 119, 'AT OBJ77 POS11': 90, 'AT OBJ66 POS33': 72, 'AT OBJ55 POS44': 24, 'AT OBJ00 POS66': 18, 'AT OBJ66 POS66': 6, 'AT OBJ11 POS12': 22, 'AT OBJ12 POS44': 35, 'AT OBJ77 POS44': 45, 'AT OBJ66 POS21': 63, 'AT OBJ21 POS33': 97, 'AT OBJ55 POS12': 15, 'AT OBJ22 POS55': 112, 'AT OBJ44 POS55': 104, 'AT OBJ55 POS77': 23, 'AT OBJ99 POS22': 96, 'AT OBJ33 POS44': 2, 'AT OBJ00 POS11': 102, 'AT OBJ66 POS23': 55, 'AT OBJ11 POS23': 12, 'AT OBJ99 POS23': 116, 'AT OBJ11 POS77': 10, 'AT OBJ12 POS13': 82, 'AT OBJ22 POS21': 144, 'AT OBJ88 POS23': 76, 'AT OBJ44 POS13': 130, 'AT OBJ88 POS22': 32, 'AT OBJ55 POS22': 38, 'AT OBJ11 POS66': 47, 'AT OBJ22 POS13': 25, 'AT OBJ99 POS55': 153, 'AT OBJ77 POS21': 37, 'AT OBJ23 POS12': 34, 'AT OBJ21 POS11': 30, 'AT OBJ00 POS33': 131, 'AT OBJ00 POS55': 136, 'AT OBJ21 POS77': 53, 